# Lecture 22 (Working with text)

## Exercise 22.1 (reading csv)

Extend your code for Handin 9 (maximum flow) to read the graphs from CSV files. For 19.2(a) the first line should contain the names of the source and sink nodes, whereas for 19.2(b) the first line should contain the name of the source, sink, and the flow value.

Input to 19.2(a):

    A,F
    A,B,3
    A,C,4
    B,D,1
    B,E,3
    C,D,1
    C,E,1
    D,F,5
    E,D,3
    E,F,1

Input to 19.2(b)

    A,F,4
    A,B,3,1
    A,C,4,1
    B,D,1,1
    B,E,3,1
    C,D,1,1
    C,E,1,1
    D,F,5,2
    E,D,3,0
    E,F,1,1

In [1]:
import csv
import numpy as np
from scipy.optimize import linprog

FLOW_FILE = 'max_flow.csv'
COST_FLOW_FILE = 'min_cost_flow.csv'

with open(FLOW_FILE) as f:
    reader = csv.reader(f)
    source, sink = next(reader)
    edges = [(u, v, int(weight)) for u, v, weight in reader]

# 19.2(a) Minimum flow

edge_from, edge_to, capacity = zip(*edges)

nodes = sorted(list(set(edge_from + edge_to)))
n = len(nodes)
node = {label: idx for label, idx in zip(nodes, range(n))}

sinks = np.array([1 if t==sink else 0 for t in edge_to])

conservation = np.zeros((n, len(edges)))

for idx, (f, t) in enumerate(zip(edge_from, edge_to)):
    conservation[node[f]][idx] = 1.0
    conservation[node[t]][idx] = -1.0

conservation = np.delete(conservation, [node[source], node[sink]], axis=0)

res = linprog(
    -sinks,
    A_eq=conservation,
    b_eq=np.zeros(conservation.shape[0]),
    A_ub=np.eye(len(edges)),
    b_ub=capacity
)

print('Minimum flow')
print(res)

# 19.2(b) Minimum cost flow

with open(COST_FLOW_FILE) as f:
    reader = csv.reader(f)
    source, sink, flow = next(reader)
    flow = int(flow)
    edges = [(u, v, int(weight), int(cost)) for u, v, weight, cost in reader]

edge_from, edge_to, capacity, cost = zip(*edges)

nodes = sorted((set(edge_from + edge_to)))
n = len(nodes)
node = {label: idx for label, idx in zip(nodes, range(n))}

sinks = np.array([1 if t==sink else 0 for t in edge_to])

conservation = np.zeros((n, len(edges)))

for idx, (f, t) in enumerate(zip(edge_from, edge_to)):
    conservation[node[f]][idx] = 1.0
    conservation[node[t]][idx] = -1.0

conservation = np.delete(conservation, [node[source], node[sink]], axis=0)

A_eq = np.append(conservation, np.array([sinks]), axis=0)
b_eq = np.append(np.zeros(conservation.shape[0]), flow)

res = linprog(
    cost,
    A_eq=A_eq,
    b_eq=b_eq,
    A_ub=np.eye(len(edges)),
    b_ub=capacity
)

print()
print('Minimum cost flow')
print(res)

Minimum flow
        message: Optimization terminated successfully. (HiGHS Status 7: Optimal)
        success: True
         status: 0
            fun: -5.0
              x: [ 3.000e+00  2.000e+00  1.000e+00  2.000e+00  1.000e+00
                   1.000e+00  4.000e+00  2.000e+00  1.000e+00]
            nit: 1
          lower:  residual: [ 3.000e+00  2.000e+00  1.000e+00  2.000e+00
                              1.000e+00  1.000e+00  4.000e+00  2.000e+00
                              1.000e+00]
                 marginals: [ 0.000e+00  0.000e+00  0.000e+00  0.000e+00
                              0.000e+00  0.000e+00  0.000e+00  0.000e+00
                              0.000e+00]
          upper:  residual: [       inf        inf        inf        inf
                                    inf        inf        inf        inf
                                    inf]
                 marginals: [ 0.000e+00  0.000e+00  0.000e+00  0.000e+00
                              0.000e+00  0.000e+00  0.

## Exercise 22.2 (re.split)

In this exercise you should use the `split` method in the module `re` to split a string into a list of substrings,
based on a regular expression capturing the set of separator symbols `!`, `?`, `;`, `:`, `.` and `,`, where a separator string is one or more consecutive of these symbols together with zero or more white space before and after the separator. An example is the below.

    'this.was a test!, and here comes   :   the rest'
    ['this', 'was a test', 'and here comes', 'the rest']

_Hint_. See the documentation for module [regular expressions](https://docs.python.org/3/library/re.html).

In [ ]:
import re

text = 'this.was a test!, and here comes   :   the rest'

L = re.split(r'\s*[!?;:,.]+\s*', text)

print(L)  # ['this', 'was a test', 'and here comes', 'the rest']

## Exercise 22.3 (regular expression)

In exercise 16.2 (missing spaces) the problem statement includes the below lines. Explain what the following lines do.

    words = [w.split(';')[0] for w in words]
    words = {re.sub('^[1-9][.] |[ .]', '', w).lower() for w in words}

In [ ]:
# words.txt from
# https://dsn.dk/wp-content/uploads/2021/03/RO2012.opslagsord.med_.homnr_.og_.ordklasse.zip

WORDS_FILE = 'words.txt'

import re
with open(WORDS_FILE, encoding='utf-8') as f:
     words = f.readlines()

print(words[:30])  # includes "aber dabei;sb."

# remove ";" and trailing text
words = [ w.split(';')[0] for w in words ]
# remove all prefixs "1. ", "2. " etc. and spaces and periods
words = { re.sub(r'^[1-9][.] +|[ .]', '', w).lower() for w in words }

print(sorted(words)[:30])

## Exercise 22.4 (find frequent phrases)

Find all occurrences of the form "**the _`word`_**" in the file [saxo.txt](http://www.gutenberg.org/ebooks/1150.txt.utf-8) (see Exercise 4.8) using a regular expression (convert the string to lower case before searching). Generate a list of the most frequent occurrences, like the below:

    410 the king
    138 the danes
    121 the enemy
    110 the same
     87 the other
     83 the first
     80 the most
     64 the sword
     63 the battle
     62 the sea
     61 the man

_Note_. There might me a line break between `the` and _`word`_ (e.g. `the king` only occurs 387 times on a single line).